# XLA – Huấn Luyện YOLOv8-Pose Phát Hiện 4 Góc Thẻ CCCD & GPLX

**Môi trường khuyến nghị:** Google Colab GPU (T4 hoặc A100)
**Mục tiêu:** Huấn luyện mô hình YOLOv8n-Pose định vị chính xác 4 góc vật lý (Top-Left, Top-Right, Bottom-Right, Bottom-Left) phục vụ biến đổi phối cảnh (Perspective Rectification) cho pipeline OCR.

## 1. Kiểm tra phần cứng GPU & Cài đặt môi trường

In [ ]:
# Kiểm tra thông tin GPU
!nvidia-smi

In [ ]:
# Cài đặt các thư viện cần thiết
!pip install -q ultralytics opencv-python pyyaml

## 2. Kết nối Google Drive & Clone / Copy Mã Nguồn XLA

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Thiết lập thư mục làm việc (tuỳ chỉnh theo đường dẫn Drive của bạn)
# %cd /content/drive/MyDrive/XLA/cccd-reader-api-main

## 3. Nhập và Chuẩn Hóa Dataset từ Roboflow

Upload file zip dataset tải từ Roboflow (định dạng YOLOv8 Pose) vào Colab, sau đó chạy kịch bản import:

In [ ]:
# Giả sử bạn upload file dataset.zip vào /content/dataset.zip
!python scripts/pose/import_roboflow.py --zip /content/dataset.zip --dest datasets/document_pose

## 4. Kiểm Định Tính Toàn Vẹn Dataset (Keypoint Validation)

In [ ]:
!python scripts/pose/validate_pose_dataset.py --data-dir datasets/document_pose --yaml datasets/document_pose/card_pose.yaml

## 5. Trực Quan Hóa Nhãn Kiểm Tra (Sanity Check)

In [ ]:
!python scripts/pose/visualize_pose_labels.py --data-dir datasets/document_pose --split train --num 3 --output-dir /content/debug_labels

## 6. Huấn Luyện Thử Nghiệm (30 Epochs)

Chạy kiểm tra pipeline huấn luyện với 30 epochs trên GPU:

In [ ]:
!python scripts/pose/train_pose.py \
  --data datasets/document_pose/card_pose.yaml \
  --model yolov8n-pose.pt \
  --profile test30 \
  --batch 16 \
  --device 0 \
  --project runs/pose \
  --name test_run_30epochs

## 7. Huấn Luyện Chính Thức (100 Epochs với Early Stopping)

In [ ]:
!python scripts/pose/train_pose.py \
  --data datasets/document_pose/card_pose.yaml \
  --model yolov8n-pose.pt \
  --profile full100 \
  --batch 16 \
  --device 0 \
  --patience 20 \
  --project runs/pose \
  --name document_pose_official

## 8. Đánh Giá Toàn Diện Mô Hình (mAP, PCK, Corner Error, Latency)

In [ ]:
!python scripts/pose/evaluate_pose.py \
  --model runs/pose/document_pose_official/weights/best.pt \
  --data datasets/document_pose/card_pose.yaml \
  --split test \
  --device 0 \
  --json-out /content/evaluation_report.json

## 9. Thử Nghiệm Trực Quan Trên Ảnh Thực Tế (Debug Tool)

In [ ]:
!python scripts/pose/debug_pose.py \
  --image datasets/document_pose/images/test/sample_test.jpg \
  --model runs/pose/document_pose_official/weights/best.pt \
  --output-dir /content/debug_output

## 10. Tải Checkpoint `best.pt` Về Máy Cục Bộ

Sao chép file trọng số tốt nhất vào Drive hoặc tải về máy để đưa vào thư mục `models/document_pose/best.pt`:

In [ ]:
from google.colab import files
files.download('runs/pose/document_pose_official/weights/best.pt')

# Hoặc sao lưu vào Drive:
# !cp runs/pose/document_pose_official/weights/best.pt /content/drive/MyDrive/XLA/best.pt